# Mejora 3: *test-time augmentation* (TTA) con fusión de cajas (WBF)

**Faster R-CNN v2 (ResNet-50 + FPN) sobre *pills-sxdht*.** Tercer experimento de mejora. A diferencia de las anteriores, **no se reentrena nada**: se toma el mejor modelo entrenado hasta ahora (por defecto, el de la mejora 1) y solo cambia **cómo se hace la inferencia**.

## Hipótesis

Tras la mejora 1, la localización sigue siendo el error más costoso: a IoU 0.85, el ΔAP de `loc` es +0.063 en test y +0.030 en validación. El modelo encuentra y clasifica casi todas las pastillas, pero sus cajas se desvían unos píxeles de la anotación.

Esa desviación tiene una parte **aleatoria**: si se le enseña al modelo la misma imagen volteada o girada, la caja que predice (una vez devuelta a la imagen original) no es exactamente la misma, y los errores de cada vista van en direcciones distintas. **Promediando las cajas de varias vistas, esos errores se compensan** y la caja media queda más cerca de la real. Es el mismo principio que promediar varias medidas ruidosas de una magnitud.

## Diseño

**Vistas.** Solo transformaciones que (a) no cambian lo que es una pastilla y (b) se pueden deshacer **de forma exacta** sobre las cajas:

| Conjunto | Vistas | Justificación |
|---|---|---|
| sin TTA | original | referencia: es el modelo de la mejora 1 tal cual |
| volteo (2) | original + volteo horizontal | la TTA más habitual |
| volteos (4) | + volteo vertical + giro de 180° | las fotos son cenitales: la pastilla no tiene "arriba" |
| D4 (8) | los 4 giros de 90° × con/sin volteo | todas las simetrías del cuadrado |

No se usan cambios de escala ni de color: la escala cambia lo que ve el modelo (redimensiona internamente a 800 px) y el color define 4 de las 7 clases.

**Fusión.** Cada vista produce sus detecciones; hay que combinarlas en una sola lista:

- **NMS**: se juntan todas y, de cada grupo de cajas solapadas, se queda la de mayor score **sin modificarla**. Puede ganar recall, pero no mejora la caja.
- **WBF** (*Weighted Boxes Fusion*, Solovyev et al., 2021): agrupa las cajas solapadas de la misma clase y las sustituye por su **media ponderada por el score**. Es la que ataca la localización. Además, una detección que aparece en pocas vistas recibe un score más bajo, lo que puede filtrar falsos positivos.

**Selección.** Las predicciones de todas las vistas se calculan una sola vez; después se evalúan todas las combinaciones (conjunto de vistas × fusión × umbral de IoU de WBF) en **validación**, y la mejor se informa en test.

| Aspecto | Mejora 1 | Mejora 3 |
|---|---|---|
| Modelo (pesos) | entrenado en la mejora 1 | **el mismo** |
| Inferencia | una pasada | **una pasada por vista + fusión** (≈ ×N en tiempo) |

## Cómo se comprueba

- **Métrica de decisión**: mAP@0.5:0.95 en validación, comparando con el mismo modelo sin TTA.
- **Hipótesis**: si el promedio de cajas funciona, la mejora debe concentrarse en mAP@0.75 y mAP@0.5:0.95, y debe bajar el ΔAP de `loc` a IoU 0.85. Con NMS, en cambio, no debería mejorar la localización.
- **Coste**: el tiempo de inferencia se multiplica por el número de vistas.

## 1. Configuración

`SOURCE_EXP` es el experimento cuyo modelo se usa. Por defecto, la mejora 1; si la mejora 2 resulta mejor en validación, se cambia a `"mejora02_augmentation_fasterrcnn"` (ese modelo se entrenó con volteos y giros, así que debería aprovechar aún mejor estas vistas).

**En Kaggle**, los pesos (`best_model.pth`, 170 MB) no están en el repositorio. Hay que añadir como entrada la salida del cuaderno que entrenó el modelo: *Add Input → Your Work → Notebooks →* el cuaderno de la mejora 1 (la versión ejecutada). El cuaderno busca el fichero en `/kaggle/input/` automáticamente.

In [ ]:
# Dependencias que pueden no estar instaladas
try:
    import datasets, torchmetrics, pycocotools  # noqa: F401
except ImportError:
    %pip install -q datasets torchmetrics pycocotools

import sys
import time
import urllib.request
from pathlib import Path

if not Path("common.py").exists():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/HectorBarro/CV1_P6/main/faster_rcnn/common.py", "common.py")
sys.path.insert(0, ".")

import matplotlib.pyplot as plt
import numpy as np
import torch
from matplotlib.patches import Rectangle
from torchvision.ops import box_iou

import common as C

%matplotlib inline

In [ ]:
EXP_NAME = "mejora03_tta_fasterrcnn"
SOURCE_EXP = "mejora01_cosine_fasterrcnn"   # modelo al que se aplica la TTA

SEED = 42
NMS_IOU = 0.5                        # umbral de NMS (el mismo que usa Faster R-CNN internamente)
WBF_IOUS = [0.5, 0.55, 0.6, 0.7]     # umbrales de agrupación de WBF que se prueban en validación

OUT_DIR = C.RESULTS_ROOT / EXP_NAME
OUT_DIR.mkdir(parents=True, exist_ok=True)


def find_weights(exp):
    # En local: resultados/<exp>/best_model.pth. En Kaggle: en la salida del cuaderno añadida como Input
    local = C.RESULTS_ROOT / exp / "best_model.pth"
    if local.exists():
        return local
    hits = sorted(Path("/kaggle/input").glob(f"**/{exp}/best_model.pth"))
    if hits:
        return hits[0]
    raise FileNotFoundError(f"No se encuentran los pesos de {exp}. En Kaggle: Add Input -> la versión ejecutada "
                            f"del cuaderno de {exp} (ver la explicación de esta sección).")


WEIGHTS = find_weights(SOURCE_EXP)
C.set_seed(SEED)
print("Dispositivo:", C.describe_device())
print("Pesos:", WEIGHTS)

## 2. Datos y modelo

Mismas particiones y limpieza de clases que en el baseline. Solo se usan validación (para elegir la configuración) y test (para informar). El modelo es Faster R-CNN v2 con los pesos de `SOURCE_EXP`; no se entrena.

In [ ]:
hf, CLASSES = C.load_splits()
ds = C.make_datasets(hf)

model = C.build_model(len(CLASSES))
model.load_state_dict(torch.load(WEIGHTS, map_location=C.device))
model.eval()

source = C.load_experiment(SOURCE_EXP)
baseline = C.load_experiment("baseline_fasterrcnn")
print(f"Imágenes -> valid: {len(ds['valid'])} | test: {len(ds['test'])}")
print(f"Modelo de {SOURCE_EXP}: val mAP {source['results']['val_mAP']:.4f} | "
      f"test mAP {source['results']['test']['mAP@0.5:0.95']:.4f} (sin TTA)")

## 3. Las vistas y cómo se deshacen

Cada vista es un par `(k, volteo)`: girar la imagen `k`·90° en sentido antihorario y, después, voltearla en horizontal si `volteo` es cierto. Con `k` = 0..3 y volteo sí/no salen las 8 simetrías del cuadrado (grupo D4).

El modelo predice cajas **sobre la vista**, así que hay que devolverlas a la imagen original deshaciendo las operaciones en orden inverso: primero el volteo (`x → ancho − x`) y después cada giro (`(x, y) → (alto − y, x)`). Como los giros son de 90° exactos, **una caja alineada con los ejes sigue siendo exactamente la misma caja**, sin agrandarse.

La celda siguiente lo comprueba con las cajas reales de una imagen: se transforman como la imagen, se deshacen, y deben coincidir con las originales al píxel.

In [ ]:
VIEWS_D4 = [(k, f) for f in (False, True) for k in range(4)]
VIEW_SETS = {
    "sin TTA": [(0, False)],
    "volteo (2)": [(0, False), (0, True)],
    "volteos (4)": [(0, False), (0, True), (2, False), (2, True)],   # giro 180° + volteo h = volteo v
    "D4 (8)": VIEWS_D4,
}


def apply_view(img, k, flip):
    x = torch.rot90(img, k, dims=(-2, -1))
    return x.flip(-1) if flip else x


def invert_boxes(boxes, k, flip, view_h, view_w):
    # Cajas [x1, y1, x2, y2] predichas sobre la vista -> coordenadas de la imagen original
    b = boxes.clone()
    if flip:  # deshacer el volteo horizontal
        b = torch.stack([view_w - b[:, 2], b[:, 1], view_w - b[:, 0], b[:, 3]], 1)
    h, w = view_h, view_w
    for _ in range(k % 4):  # deshacer cada giro de 90° antihorario
        b = torch.stack([h - b[:, 3], b[:, 0], h - b[:, 1], b[:, 2]], 1)
        h, w = w, h
    return b


# Comprobación: transformar las cajas reales como la imagen (con una máscara) y deshacer la vista
img, tgt = ds["test"][0]
H, W = img.shape[-2:]
max_err = 0.0
for k, f in VIEWS_D4:
    view_boxes = []
    for x1, y1, x2, y2 in tgt["boxes"].round().int().tolist():
        mask = torch.zeros(1, H, W)
        mask[0, y1:y2, x1:x2] = 1
        ys, xs = torch.nonzero(apply_view(mask, k, f)[0], as_tuple=True)
        view_boxes.append([xs.min(), ys.min(), xs.max() + 1, ys.max() + 1])
    v = apply_view(img, k, f)
    back = invert_boxes(torch.tensor(view_boxes, dtype=torch.float32), k, f, v.shape[-2], v.shape[-1])
    max_err = max(max_err, (back - tgt["boxes"].round()).abs().max().item())
print(f"Error máximo al deshacer las 8 vistas: {max_err:.1f} píxeles")

## 4. Predicciones de todas las vistas

Se pasa cada imagen de validación y test por el modelo en sus 8 vistas (en un solo lote por imagen) y se guardan las detecciones ya devueltas a la imagen original. A partir de aquí, cualquier conjunto de vistas y cualquier fusión se evalúan sin volver a ejecutar el modelo.

In [ ]:
@torch.no_grad()
def predict_views(img, views):
    # Detecciones de cada vista, en coordenadas de la imagen original
    x = img.to(C.device)
    batch = [apply_view(x, k, f) for k, f in views]
    outs = model(batch)
    return [{"boxes": invert_boxes(o["boxes"].cpu(), k, f, v.shape[-2], v.shape[-1]),
             "scores": o["scores"].cpu(), "labels": o["labels"].cpu()}
            for (k, f), v, o in zip(views, batch, outs)]


raw, targets = {}, {}
for split in ["valid", "test"]:
    t0 = time.time()
    raw[split], targets[split] = [], []
    for i in range(len(ds[split])):
        img, tgt = ds[split][i]
        raw[split].append(dict(zip(VIEWS_D4, predict_views(img, VIEWS_D4))))
        targets[split].append({"boxes": tgt["boxes"].as_subclass(torch.Tensor), "labels": tgt["labels"]})
    print(f"{split}: {len(raw[split])} imágenes x {len(VIEWS_D4)} vistas en {time.time() - t0:.0f} s")

### 4.1 Cómo se ve la fusión en una pastilla

Para entender qué hace cada fusión, se muestran algunas pastillas de **validación** ampliadas, con la caja real (blanco), la caja de cada una de las 8 vistas (líneas finas), el resultado de NMS (azul discontinuo) y el de WBF (rojo). Se eligen las pastillas donde el modelo sin TTA tiene la caja **peor ajustada**, que es donde la fusión puede ayudar.

- NMS se queda con una de las cajas de las vistas, así que su IoU con la real es el de esa vista.
- WBF promedia: si las vistas se equivocan hacia lados distintos, la media queda más cerca de la caja real.

In [ ]:
def fuse(preds_by_view, views, method, iou):
    preds = [preds_by_view[v] for v in views]
    if method == "ninguna" or len(views) == 1:
        return preds[0]
    return C.wbf(preds, iou_thr=iou) if method == "WBF" else C.nms_fusion(preds, iou_thr=iou)


def best_match_iou(pred, box, label):
    # IoU de la detección de la clase correcta que más se solapa con la caja real
    m = pred["labels"] == label
    return box_iou(box[None], pred["boxes"][m]).max().item() if m.any() else 0.0


cases = []
for i, (r, t) in enumerate(zip(raw["valid"], targets["valid"])):
    for box, label in zip(t["boxes"], t["labels"]):
        cases.append((best_match_iou(r[(0, False)], box, label), i, box, label))
cases = sorted(cases, key=lambda c: c[0])[:3]

fig, axes = plt.subplots(1, 3, figsize=(16, 5.6))
for ax, (iou0, i, box, label) in zip(axes, cases):
    img = ds["valid"][i][0].permute(1, 2, 0).numpy()
    x1, y1, x2, y2 = box.tolist()
    pad = 0.35 * max(x2 - x1, y2 - y1)
    ax.imshow(img)
    for v in VIEWS_D4:
        p = raw["valid"][i][v]
        m = (p["labels"] == label) & (p["scores"] >= 0.3)
        if m.any():
            bx = p["boxes"][m][box_iou(box[None], p["boxes"][m])[0].argmax()].tolist()
            ax.add_patch(Rectangle(bx[:2], bx[2] - bx[0], bx[3] - bx[1], fill=False, lw=0.8, color="gold", alpha=0.8))
    ax.add_patch(Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, lw=2.5, color="white"))
    res = {}
    for method, color, ls in [("NMS", "deepskyblue", "--"), ("WBF", "red", "-")]:
        fused = fuse(raw["valid"][i], VIEWS_D4, method, 0.55 if method == "WBF" else NMS_IOU)
        m = fused["labels"] == label
        if not m.any():
            res[method] = 0.0
            continue
        b = fused["boxes"][m][box_iou(box[None], fused["boxes"][m])[0].argmax()]
        res[method] = box_iou(box[None], b[None]).item()
        bx = b.tolist()
        ax.add_patch(Rectangle(bx[:2], bx[2] - bx[0], bx[3] - bx[1], fill=False, lw=2, color=color, ls=ls))
    ax.set_xlim(x1 - pad, x2 + pad)
    ax.set_ylim(y2 + pad, y1 - pad)
    ax.set_title(f"valid #{i} · {CLASSES[label]}\nIoU con la real: sin TTA {iou0:.3f} | NMS {res['NMS']:.3f} | "
                 f"WBF {res['WBF']:.3f}", fontsize=9)
    ax.axis("off")
fig.suptitle("Blanco: caja real · amarillo: 8 vistas · azul discontinuo: NMS · rojo: WBF (D4, IoU 0.55)", fontsize=10)
plt.tight_layout()
plt.savefig(OUT_DIR / "fusion_ejemplos.png", dpi=110)
plt.show()

## 5. Elección de la configuración en validación

Se evalúan todas las combinaciones de conjunto de vistas, método de fusión y umbral de IoU de WBF. Se elige la de **mayor mAP@0.5:0.95 en validación**; la columna de test se muestra solo como información y **no se usa para elegir**.

Hay que tener en cuenta que la validación tiene solo 45 imágenes y que se prueban unas 20 configuraciones: la mejor en validación está algo favorecida por el azar, así que diferencias de pocas décimas de punto entre configuraciones no son significativas. Por eso conviene fijarse también en si la tendencia es coherente (por ejemplo, si WBF supera a NMS con todos los conjuntos de vistas).

In [ ]:
configs = [("sin TTA", "ninguna", None)]
for vs in ["volteo (2)", "volteos (4)", "D4 (8)"]:
    configs.append((vs, "NMS", NMS_IOU))
    configs += [(vs, "WBF", iou) for iou in WBF_IOUS]

rows = []
for vs, method, iou in configs:
    row = {"vistas": vs, "fusión": method, "iou": iou}
    for split in ["valid", "test"]:
        preds = [fuse(r, VIEW_SETS[vs], method, iou) for r in raw[split]]
        m = C.evaluate_predictions(preds, targets[split])
        row[split] = {"mAP": m["map"].item(), "mAP50": m["map_50"].item(), "mAP75": m["map_75"].item()}
    rows.append(row)

print(f"{'vistas':<13}{'fusión':<9}{'IoU':>5} | {'val mAP':>8}{'val @50':>8}{'val @75':>8} | {'test mAP':>9}{'test @75':>9}")
for r in rows:
    iou = f"{r['iou']:.2f}" if r["iou"] else "-"
    print(f"{r['vistas']:<13}{r['fusión']:<9}{iou:>5} | {r['valid']['mAP']:>8.4f}{r['valid']['mAP50']:>8.4f}"
          f"{r['valid']['mAP75']:>8.4f} | {r['test']['mAP']:>9.4f}{r['test']['mAP75']:>9.4f}")

best = max(rows, key=lambda r: r["valid"]["mAP"])
BEST_VIEWS, BEST_METHOD, BEST_IOU = VIEW_SETS[best["vistas"]], best["fusión"], best["iou"]
print(f"\nElegida en validación: {best['vistas']} + {best['fusión']}"
      + (f" (IoU {best['iou']})" if best["iou"] else "") + f" -> val mAP {best['valid']['mAP']:.4f}")

In [ ]:
# Resumen gráfico: mejor umbral de WBF y NMS por conjunto de vistas, en validación y test
fig, axes = plt.subplots(1, 2, figsize=(14, 4.2), sharey=True)
names = ["volteo (2)", "volteos (4)", "D4 (8)"]
no_tta = rows[0]
for ax, split in zip(axes, ["valid", "test"]):
    x = np.arange(len(names))
    nms = [next(r for r in rows if r["vistas"] == n and r["fusión"] == "NMS")[split]["mAP"] for n in names]
    wbf_best = [max(r[split]["mAP"] for r in rows if r["vistas"] == n and r["fusión"] == "WBF") for n in names]
    ax.bar(x - 0.18, nms, 0.36, label="NMS")
    ax.bar(x + 0.18, wbf_best, 0.36, label="WBF (mejor umbral)")
    ax.axhline(no_tta[split]["mAP"], color="gray", ls="--", label="sin TTA")
    ax.set_xticks(x, names)
    ax.set_title(f"mAP@0.5:0.95 en {split}")
    lo = min(min(nms), min(wbf_best), no_tta[split]["mAP"])
    ax.set_ylim(lo - 0.03, None)
    ax.grid(axis="y", alpha=0.3)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "tta_configuraciones.png", dpi=120)
plt.show()

## 6. Evaluación en test

Con la configuración elegida en validación: mAP de COCO, AP por clase y curvas PR, igual que la sección 10 del baseline.

Se comprueba también que la fila "sin TTA" reproduce el resultado de test de la mejora 1: es el mismo modelo con una sola vista, así que debe coincidir.

**Tiempo de inferencia.** Con TTA, cada imagen pasa por el modelo una vez por vista (en un solo lote) y después se fusionan las detecciones. Se mide el tiempo total por imagen con y sin TTA en el mismo hardware.

In [ ]:
print(f"Sin TTA en test: {rows[0]['test']['mAP']:.4f} | resultado de {SOURCE_EXP}: "
      f"{source['results']['test']['mAP@0.5:0.95']:.4f}\n")

final = {split: [fuse(r, BEST_VIEWS, BEST_METHOD, BEST_IOU) for r in raw[split]] for split in ["valid", "test"]}
res = C.evaluate_predictions(final["test"], targets["test"], extended=True)
res, per_class = C.results_report(res, CLASSES)
C.plot_pr_curves(res, CLASSES, out_path=OUT_DIR / "curvas_pr_test.png")


def tta_predict(img):
    return fuse(dict(zip(BEST_VIEWS, predict_views(img, BEST_VIEWS))), BEST_VIEWS, BEST_METHOD, BEST_IOU)


times_ms = C.measure_inference(None, ds["test"], predict=tta_predict)
times_base = C.measure_inference(model, ds["test"])
print(f"\nHardware: {C.hardware_name()}")
print(f"Inferencia sin TTA : {times_base.mean():.1f} ± {times_base.std():.1f} ms/imagen")
print(f"Inferencia con TTA : {times_ms.mean():.1f} ± {times_ms.std():.1f} ms/imagen "
      f"({len(BEST_VIEWS)} vistas, x{times_ms.mean() / times_base.mean():.1f})")

### 6.1 Comparación

Entre paréntesis, la diferencia con la primera columna (baseline). El efecto de la TTA es la diferencia entre la mejora 3 y el modelo de origen sin TTA, que tiene exactamente los mismos pesos. El tiempo de entrenamiento es el del modelo de origen, porque la TTA no entrena.

In [ ]:
config = {"modelo_origen": SOURCE_EXP, "vistas": best["vistas"], "fusion": BEST_METHOD,
          "iou_fusion": BEST_IOU, "n_vistas": len(BEST_VIEWS)}
fit_result = {"best_epoch": source["results"]["mejor_epoca"], "best_map": best["valid"]["mAP"],
              "train_time": source["results"]["tiempo_entrenamiento_min"] * 60}
DESCRIPTION = f"TTA {best['vistas']} + {BEST_METHOD} sobre el modelo de {SOURCE_EXP}"
summary = C.make_summary(EXP_NAME, DESCRIPTION, config, fit_result, res, per_class, times_ms)
summary["tta_configuraciones"] = rows

source_label = "mejora 1" if SOURCE_EXP.startswith("mejora01") else SOURCE_EXP
C.compare_metrics({"baseline": baseline["results"], f"{source_label} (sin TTA)": source["results"],
                   "mejora 3 (TTA)": summary})

### 6.2 Análisis cualitativo

Como en la sección 12 del baseline: las mismas 6 imágenes de test, con las cajas reales (**verde**) y las predicciones con TTA con score ≥ 0.5 (**rojo**). Se pueden comparar una a una con `ejemplos_test.png` del baseline y de la mejora 1.

In [ ]:
C.plot_examples(None, ds["test"], CLASSES, OUT_DIR / "ejemplos_test.png", seed=SEED, predict=tta_predict)

## 7. Análisis de errores

Se repite el análisis de la sección 13 del baseline con las predicciones con TTA, con las mismas figuras, comparando siempre con el **mismo modelo sin TTA** (calculado en este cuaderno): descomposición TIDE (7.1), AP en función del umbral de IoU (7.2), matriz de confusión (7.3) e imágenes con más errores (7.4).

También se guardan las predicciones con TTA de validación y test (`preds_valid.json`, `preds_test.json`), para la comparativa final y el ensemble.

### 7.1 Descomposición TIDE

Como en la sección 13.3 del baseline, a IoU 0.5 y 0.85. Si el promedio de cajas funciona, **el ΔAP de `loc` a IoU 0.85 debe bajar** respecto al modelo sin TTA. `miss` no debería cambiar en test (son las 2 cajas mal anotadas de la imagen 74).

In [ ]:
split_data = {s: C.predictions_to_mapviz(final[s], targets[s], CLASSES) for s in ["valid", "test"]}
no_tta = {s: C.predictions_to_mapviz([r[(0, False)] for r in raw[s]], targets[s], CLASSES) for s in ["valid", "test"]}
C.save_predictions(split_data, OUT_DIR)

tide = C.tide_analysis(split_data)
tide_no_tta = C.tide_analysis(no_tta, verbose=False)
print()
C.compare_tide({"baseline": baseline["tide"], f"{source_label} (sin TTA)": C.tide_to_json(tide_no_tta),
                "mejora 3 (TTA)": C.tide_to_json(tide)}, keys=("valid@0.85", "test@0.85", "test@0.5"))

In [ ]:
C.plot_tide(tide, OUT_DIR / "tide.png")

### 7.2 AP en función del umbral de IoU

Como en la sección 13.2 del baseline, superponiendo el mismo modelo sin TTA. Si el promedio de cajas mejora la localización, la curva con TTA debe **caer más tarde** (a IoU más altos) y el histograma de IoU de los aciertos debe desplazarse **hacia 1**. A IoU bajos las dos curvas deberían coincidir.

In [ ]:
C.plot_ap_vs_iou(split_data, "mejora 3 (TTA)", OUT_DIR, references={"sin TTA": no_tta})

### 7.3 Matriz de confusión

Como en la sección 13.4 del baseline (score ≥ 0.5, IoU ≥ 0.5). Con WBF, una detección que solo aparece en algunas vistas recibe un score más bajo, así que la fila *fondo* (falsos positivos, como los lunares de la imagen 81) podría bajar. La columna *no detectado* de test son las cajas mal anotadas de la imagen 74.

In [ ]:
cms = C.plot_confusion(split_data, CLASSES, OUT_DIR / "matriz_confusion.png")
C.print_confusion_summary(cms, CLASSES)

### 7.4 Imágenes con más errores

Como en la sección 13.5 del baseline: imágenes de test con más errores a IoU 0.5 y score ≥ 0.3. Blanco = caja real, ámbar discontinuo = no detectada, verde = acierto, y el resto de colores según el tipo de error.

In [ ]:
gts_test, preds_test = split_data["test"]
C.plot_worst_images(gts_test, preds_test, tide["test@0.5"], ds["test"], OUT_DIR / "peores_imagenes_test.png")

## 8. Guardar resultados

`results.json` (incluye la tabla de todas las configuraciones probadas), `tide.json` y las predicciones quedan en `faster_rcnn/resultados/mejora03_tta_fasterrcnn/`.

In [ ]:
C.save_experiment(OUT_DIR, summary, tide)

if Path("/kaggle/working").exists():
    import shutil
    from IPython.display import FileLink

    shutil.make_archive(f"/kaggle/working/{EXP_NAME}", "zip", OUT_DIR)
    display(FileLink(f"{EXP_NAME}.zip"))

## 9. Conclusiones

> **Completar tras ejecutar el cuaderno**, con los números de las secciones 5, 6.1 y 7:
>
> - ¿Qué configuración gana en validación? ¿WBF supera a NMS de forma consistente (con todos los conjuntos de vistas)?
> - ¿La ganancia frente al modelo sin TTA se concentra en mAP@0.75 / mAP@0.5:0.95? ¿Baja el ΔAP de `loc` a IoU 0.85?
> - ¿Cambian los falsos positivos de fondo (imagen 81) con WBF?
> - Coste: ¿cuánto se multiplica el tiempo de inferencia? ¿Compensa para una aplicación real?